<a href="https://colab.research.google.com/github/manya28/LMEs/blob/main/LME.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# install.packages(c("nlme", "JMbayes2", "survival", "lattice"))

# Load necessary libraries
library(nlme)       # For Linear Mixed Effects Model (LME)
library(JMbayes2)         # For Joint Modelling
library(survival)   # For Cox model
library(dplyr)  # or library(magrittr)

# Read CSV files into R
train_data <- read.csv("train_data.csv")
test_data  <- read.csv("test_data.csv")
val_data   <- read.csv("val_data.csv")

# Check structure of the data
str(train_data)
train_data <- subset(train_data,select = -X)
head(train_data)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message in install.packages(c("nlme", "JMbayes2", "survival", "lattice")):
“installation of package ‘survival’ had non-zero exit status”


'data.frame':	16251 obs. of  16 variables:
 $ X              : int  0 1 15 17 18 19 20 21 22 60 ...
 $ pid            : int  200143 200551 200701 200800 200817 201005 201005 201005 201005 201519 ...
 $ base_age       : num  7.22 11.21 7.13 7.24 16.13 ...
 $ sex            : num  1 0 0 0 0 0 0 0 0 0 ...
 $ tstart         : num  0 0 0 0 0 ...
 $ tstop          : num  0.7556 0.9829 0.0192 0.4298 0.0438 ...
 $ t1d            : int  0 1 1 1 1 0 0 0 0 0 ...
 $ aabStatus      : int  1 0 1 1 1 1 1 1 1 0 ...
 $ c_pep_auc      : num  7.11 2.31 3.71 2.86 6.95 ...
 $ hba1c          : num  4.9 6 5.4 5.6 6.1 4.8 4.8 4.8 5 5 ...
 $ fasting_glucose: num  103 97 94 87 123 81 81 81 88 78 ...
 $ early_pep_delta: num  7.27 0.87 2.31 2.54 2.12 ...
 $ astart         : num  7.22 11.21 7.13 7.24 16.13 ...
 $ astop          : num  7.97 12.19 7.15 7.67 16.17 ...
 $ glu_2hr        : num  121 308 245 192 290 108 108 108 98 112 ...
 $ time_int       : num  0.7556 0.9829 0.0192 0.4298 0.0438 ...


,pid,base_age,sex,tstart,tstop,t1d,aabStatus,c_pep_auc,hba1c,fasting_glucose,early_pep_delta,astart,astop,glu_2hr,time_int
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,200143,7.216290,1,0,0.75564682,0,1,7.114583,4.9,103,7.275,7.216290,7.971937,121,0.75564682
2,200551,11.205339,0,0,0.98288843,1,0,2.313333,6.0,97,0.870,11.205339,12.188227,308,0.98288843
3,200701,7.128679,0,0,0.01916496,1,1,3.712917,5.4,94,2.315,7.128679,7.147844,245,0.01916496
4,200800,7.240931,0,0,0.42984257,1,1,2.858750,5.6,87,2.545,7.240931,7.670773,192,0.42984257
5,200817,16.128679,0,0,0.04380561,1,1,6.952917,6.1,123,2.125,16.128679,16.172485,290,0.04380561
6,201005,18.251882,0,0,0.74469541,0,1,5.190417,4.8,81,3.175,18.251882,18.996578,108,0.74469541


In [7]:
surv_data <- train_data %>%
  group_by(pid) %>%
  summarise(
    base_age = first(base_age),
    sex = first(sex),
    aabStatus = first(aabStatus),
    tstop = max(as.numeric(tstop)),   # using the maximum 'tstop' as the follow-up time
    t1d = max(t1d)                     # event indicator should be consistent across rows per patient
  )

In [8]:
surv_fit <- coxph(Surv(tstop, t1d) ~ base_age + aabStatus,
                  data = surv_data,
                  x = TRUE)

In [9]:
surv_fit

Call:
coxph(formula = Surv(tstop, t1d) ~ base_age + aabStatus, data = surv_data, 
    x = TRUE)

               coef exp(coef)  se(coef)      z        p
base_age  -0.016520  0.983616  0.004665 -3.541 0.000399
aabStatus  2.126192  8.382883  0.151413 14.042  < 2e-16

Likelihood ratio test=360.1  on 2 df, p=< 2.2e-16
n= 2687, number of events= 428 

In [10]:
lme_med <- lme(glu_2hr ~ astart + base_age + hba1c + fasting_glucose + aabStatus,
                  random = ~ tstart | pid,
                  data = train_data)

# Longitudinal model for c_pep_auc (also known to affect T1D staging)
lme_large_1 <- lme(c_pep_auc ~ astart + hba1c + fasting_glucose + aabStatus + glu_2hr,
                random = ~ tstart | pid,
                data = train_data)

lme_large_2 <- lme(early_pep_delta ~ astart + hba1c + fasting_glucose + aabStatus + glu_2hr,
                    random = ~ tstart | pid,
                    data = train_data)

In [21]:
# Increase the number of iterations, burn-in, and thinning in the jm() function.
joint_model_med <- jm(surv_fit, lme_med, time_var = "tstop",
                     control = list(n_iter = 3000, n_burnin = 1000, n_thin = 7))
   # This gives the model more chances to converge.

# Check the summary of the joint model. The association parameters (often denoted alpha)
# will indicate the strength of each biomarker's effect on the hazard of T1D onset.
joint_model_large_cpep <- jm(surv_fit, lme_large_1, time_var = "tstop",
                     control = list(n_iter = 3000, n_burnin = 1000, n_thin = 7))

joint_model_large_auc <- jm(surv_fit, lme_large_2, time_var = "tstop",
                     control = list(n_iter = 3000, n_burnin = 1000, n_thin = 7))

In [22]:
summary(joint_model_med)


Call:
jm(Surv_object = surv_fit, Mixed_objects = lme_med, time_var = "tstop", 
    control = list(n_iter = 3000, n_burnin = 1000, n_thin = 7))

Data Descriptives:
Number of Groups: 2687		Number of events: 428 (15.9%)
Number of Observations:
  glu_2hr: 16251

                 DIC     WAIC      LPML
marginal    148500.8 148470.4 -74235.35
conditional 187817.6 186209.3 -94932.55

Random-effects covariance matrix:
                      
       StdDev    Corr 
(Intr) 37.5053 (Intr) 
tstart 7.1737  -0.8253

Survival Outcome:
                  Mean  StDev    2.5%   97.5% P   Rhat
base_age       -0.0306 0.0062 -0.0428 -0.0187 0 1.0045
aabStatus       1.7970 0.1601  1.4951  2.1137 0 1.0109
value(glu_2hr)  0.0343 0.0013  0.0320  0.0368 0 1.0329

Longitudinal Outcome: glu_2hr (family = gaussian, link = identity)
                   Mean  StDev    2.5%   97.5%      P   Rhat
(Intercept)     10.0985 5.1755  0.4616 19.8985 0.0443 1.0738
astart          -2.4700 0.1730 -2.8607 -2.1954 0.0000 3.0757
bas

In [23]:
summary(joint_model_large_cpep)


Call:
jm(Surv_object = surv_fit, Mixed_objects = lme_large_1, time_var = "tstop", 
    control = list(n_iter = 3000, n_burnin = 1000, n_thin = 7))

Data Descriptives:
Number of Groups: 2687		Number of events: 428 (15.9%)
Number of Observations:
  c_pep_auc: 16251

                 DIC     WAIC      LPML
marginal    59299.75 59268.66 -29629.28
conditional 68815.52 67250.66 -35492.35

Random-effects covariance matrix:
                     
       StdDev   Corr 
(Intr) 2.4809 (Intr) 
tstart 0.3378 -0.3986

Survival Outcome:
                    Mean  StDev    2.5%   97.5%      P   Rhat
base_age         -0.0030 0.0068 -0.0170  0.0092 0.6713 1.0141
aabStatus         2.0521 0.1692  1.7311  2.3814 0.0000 1.0132
value(c_pep_auc) -0.1650 0.0299 -0.2245 -0.1089 0.0000 1.0622

Longitudinal Outcome: c_pep_auc (family = gaussian, link = identity)
                   Mean  StDev    2.5%   97.5%      P   Rhat
(Intercept)      0.2422 0.3694 -0.4814  0.9537 0.5268 1.1090
astart           0.0781 0.0040  

In [24]:
summary(joint_model_large_auc)


Call:
jm(Surv_object = surv_fit, Mixed_objects = lme_large_2, time_var = "tstop", 
    control = list(n_iter = 3000, n_burnin = 1000, n_thin = 7))

Data Descriptives:
Number of Groups: 2687		Number of events: 428 (15.9%)
Number of Observations:
  early_pep_delta: 16251

                 DIC     WAIC      LPML
marginal    62468.30 62448.72 -31225.25
conditional 72785.98 71199.70 -37467.54

Random-effects covariance matrix:
                     
       StdDev   Corr 
(Intr) 2.2402 (Intr) 
tstart 0.3314 -0.4179

Survival Outcome:
                          Mean  StDev    2.5%   97.5%      P   Rhat
base_age               -0.0064 0.0063 -0.0193  0.0055 0.3077 1.0056
aabStatus               1.8440 0.1686  1.5339  2.1966 0.0000 1.0238
value(early_pep_delta) -0.5282 0.0448 -0.6178 -0.4434 0.0000 1.0214

Longitudinal Outcome: early_pep_delta (family = gaussian, link = identity)
                   Mean  StDev    2.5%   97.5%      P   Rhat
(Intercept)      3.5747 0.3652  2.8589  4.2550 0.0000 1.0

In [25]:
# Organize the longitudinal models in a list.
lme_list <- list(
  glu_2hr = lme_med,
  c_pep_auc = lme_large_1,
  early_pep_delta = lme_large_2
)

In [26]:
joint_model <- jm(surv_fit, lme_list, time_var = "tstop",
                  control = list(n_iter = 3000, n_burnin = 1000, n_thin = 7))

In [27]:
summary(joint_model)


Call:
jm(Surv_object = surv_fit, Mixed_objects = lme_list, time_var = "tstop", 
    control = list(n_iter = 3000, n_burnin = 1000, n_thin = 7))

Data Descriptives:
Number of Groups: 2687		Number of events: 428 (15.9%)
Number of Observations:
  glu_2hr: 16251
  c_pep_auc: 16251
  early_pep_delta: 16251

                 DIC     WAIC      LPML
marginal    259933.2 260005.5 -129999.8
conditional 312708.6 308157.9 -157595.2

Random-effects covariance matrix:
                                                      
       StdDev    Corr                                 
(Intr) 37.6719 (Intr)  tstart  (Intr)  tstart  (Intr) 
tstart 7.2675  -0.8292                                
(Intr) 2.5394  -0.2287 0.2203                         
tstart 0.3759  0.1060  -0.1303 -0.4171                
(Intr) 2.3251  -0.2350 0.1809  0.9235  -0.4164        
tstart 0.3781  0.0727  -0.1581 -0.4391 0.9379  -0.4287

Survival Outcome:
                          Mean  StDev    2.5%   97.5% P   Rhat
base_age          